<a href="https://colab.research.google.com/github/muha-1680/complaint-classifier-group-project/blob/main/Copy_of_Student_Complaint_Classifier_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================================
# CELL 1: INSTALL ALL PACKAGES AND IMPORTS (FIXED)
# ============================================================================

!pip install -q torch transformers pandas numpy scikit-learn gradio matplotlib seaborn nltk tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW  # <--- FIXED: Using PyTorch's AdamW
from transformers import BertTokenizer, BertModel
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')
import gradio as gr
from google.colab import files
import time

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("="*50)
print("🚀 STUDENT COMPLAINT CLASSIFIER")
print("="*50)
print(f"✅ Device: {device}")
print(f"✅ GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU Name: {torch.cuda.get_device_name(0)}")
print("="*50)

🚀 STUDENT COMPLAINT CLASSIFIER
✅ Device: cuda
✅ GPU Available: True
✅ GPU Name: Tesla T4


In [ ]:
# ============================================================================
# CELL 2: UPLOAD YOUR CSV FILE
# ============================================================================

print("📤 PLEASE UPLOAD YOUR FILE: student_complaints_700.csv")
print("-"*50)
print("Click 'Choose Files' button below 👇")
print("-"*50)

uploaded = files.upload()

# Verify upload
df_check = pd.read_csv('student_complaints_700.csv')
print("\n" + "="*50)
print("✅ FILE UPLOADED SUCCESSFULLY!")
print("="*50)
print(f"📊 Total Complaints: {len(df_check)}")
print(f"📊 Columns: {list(df_check.columns)}")
print("\n📝 First 5 complaints:")
print(df_check.head())
print("="*50)

📤 PLEASE UPLOAD YOUR FILE: student_complaints_700.csv
--------------------------------------------------
Click 'Choose Files' button below 👇
--------------------------------------------------


Saving student_complaints_700.csv to student_complaints_700.csv

✅ FILE UPLOADED SUCCESSFULLY!
📊 Total Complaints: 700
📊 Columns: ['complaint_text', 'category']

📝 First 5 complaints:
                               complaint_text              category
0  Motorcycles move dangerously inside campus        Security Issue
1          There are no lights on the walkway        Security Issue
2  Motorcycles move dangerously inside campus        Security Issue
3                   Staff are not cooperative  Administrative Issue
4  The computer lab computers are not working             ICT Issue


In [ ]:
# ============================================================================
# CELL 3: PREPROCESSING AND DATASET CLASSES
# ============================================================================

class ComplaintPreprocessor:
    def __init__(self):
        self.label_encoder = LabelEncoder()

    def clean_text(self, text):
        """Clean complaint text"""
        text = str(text).lower()
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        text = ' '.join(text.split())
        return text

    def load_and_preprocess(self, file_path):
        """Load and preprocess data"""
        df = pd.read_csv(file_path)
        print(f"📊 Loaded {len(df)} complaints")

        # Clean text
        df['cleaned_text'] = df['complaint_text'].apply(self.clean_text)

        # Encode labels
        df['label'] = self.label_encoder.fit_transform(df['category'])

        # Show distribution
        print("\n📈 CATEGORY DISTRIBUTION:")
        print("-"*30)
        for category in self.label_encoder.classes_:
            count = len(df[df['category'] == category])
            percentage = (count/len(df))*100
            print(f"  {category:20s}: {count:3d} ({percentage:.1f}%)")

        return df

    def split_data(self, df):
        """Split into train/val/test"""
        X = df['cleaned_text'].values
        y = df['label'].values

        X_train, X_temp, y_train, y_temp = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )

        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
        )

        print("\n📊 DATA SPLIT:")
        print("-"*30)
        print(f"  Training   : {len(X_train):3d} samples ({len(X_train)/len(X)*100:.1f}%)")
        print(f"  Validation : {len(X_val):3d} samples ({len(X_val)/len(X)*100:.1f}%)")
        print(f"  Testing    : {len(X_test):3d} samples ({len(X_test)/len(X)*100:.1f}%)")

        return X_train, X_val, X_test, y_train, y_val, y_test


class ComplaintDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

print("✅ Preprocessing classes created successfully!")


✅ Preprocessing classes created successfully!


In [ ]:
# ============================================================================
# CELL 4: BERT MODEL ARCHITECTURE
# ============================================================================

class BERTComplaintClassifier(nn.Module):
    def __init__(self, n_classes, model_name='bert-base-uncased'):
        super(BERTComplaintClassifier, self).__init__()

        self.bert = BertModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, n_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        x = self.dropout(pooled_output)
        x = self.classifier(x)
        return x

print("✅ BERT Model defined successfully!")
print(f"✅ Base model: bert-base-uncased")


✅ BERT Model defined successfully!
✅ Base model: bert-base-uncased


In [ ]:
# ============================================================================
# CELL 5: LOAD DATA AND PREPARE DATALOADERS
# ============================================================================

print("="*50)
print("📊 LOADING AND PREPARING DATA")
print("="*50)

# Initialize preprocessor
preprocessor = ComplaintPreprocessor()

# Load data
df = preprocessor.load_and_preprocess('student_complaints_700.csv')

# Split data
X_train, X_val, X_test, y_train, y_val, y_test = preprocessor.split_data(df)

# Initialize tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print("\n✅ Tokenizer loaded!")

# Create datasets
train_dataset = ComplaintDataset(X_train, y_train, tokenizer)
val_dataset = ComplaintDataset(X_val, y_val, tokenizer)
test_dataset = ComplaintDataset(X_test, y_test, tokenizer)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

# Get class info
n_classes = len(preprocessor.label_encoder.classes_)
class_names = preprocessor.label_encoder.classes_

print("\n" + "="*50)
print("✅ DATA PREPARATION COMPLETE!")
print("="*50)
print(f"🎯 Number of classes: {n_classes}")
print(f"📋 Classes: {', '.join(class_names)}")
print("="*50)


📊 LOADING AND PREPARING DATA
📊 Loaded 700 complaints

📈 CATEGORY DISTRIBUTION:
------------------------------
  Academic Issue      :  80 (11.4%)
  Administrative Issue:  92 (13.1%)
  Cafeteria Issue     : 101 (14.4%)
  Dormitory Issue     :  95 (13.6%)
  Financial Issue     :  81 (11.6%)
  ICT Issue           :  84 (12.0%)
  Library Issue       :  86 (12.3%)
  Security Issue      :  81 (11.6%)

📊 DATA SPLIT:
------------------------------
  Training   : 560 samples (80.0%)
  Validation :  70 samples (10.0%)
  Testing    :  70 samples (10.0%)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


✅ Tokenizer loaded!

✅ DATA PREPARATION COMPLETE!
🎯 Number of classes: 8
📋 Classes: Academic Issue, Administrative Issue, Cafeteria Issue, Dormitory Issue, Financial Issue, ICT Issue, Library Issue, Security Issue


In [ ]:
# ============================================================================
# CELL 6: INITIALIZE MODEL AND OPTIMIZER (FIXED)
# ============================================================================

# Initialize model
model = BERTComplaintClassifier(n_classes).to(device)

# Setup optimizer and loss - FIXED: Using torch.optim.AdamW
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("="*50)
print("🚀 MODEL INITIALIZED")
print("="*50)
print(f"📊 Total parameters: {total_params:,}")
print(f"📊 Trainable parameters: {trainable_params:,}")
print(f"🎯 Number of classes: {n_classes}")
print(f"📋 Classes: {', '.join(class_names[:5])}...")
print(f"💻 Device: {device}")
print("="*50)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🚀 MODEL INITIALIZED
📊 Total parameters: 109,488,392
📊 Trainable parameters: 109,488,392
🎯 Number of classes: 8
📋 Classes: Academic Issue, Administrative Issue, Cafeteria Issue, Dormitory Issue, Financial Issue...
💻 Device: cuda


In [ ]:
# ============================================================================
# CELL 7: TRAINING LOOP
# ============================================================================

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0
EPOCHS = 5

print("="*50)
print("🎯 STARTING TRAINING")
print("="*50)
print(f"📊 Epochs: {EPOCHS}")
print(f"📊 Batch size: 16")
print(f"📊 Learning rate: 2e-5")
print("="*50)

for epoch in range(EPOCHS):
    print(f"\n📌 EPOCH {epoch + 1}/{EPOCHS}")
    print("-"*40)

    # Training phase
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    train_bar = tqdm(train_loader, desc="Training")
    for batch in train_bar:
        # Move to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Statistics
        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

        # Update progress bar
        train_bar.set_postfix({'loss': f'{loss.item():.4f}'})

    train_acc = train_correct / train_total
    train_loss = train_loss / len(train_loader)

    # Validation phase
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc="Validation")
        for batch in val_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = val_correct / val_total
    val_loss = val_loss / len(val_loader)

    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f"\n  ✅ Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  ✅ Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'model_state_dict': model.state_dict(),
            'label_encoder': preprocessor.label_encoder,
            'class_names': class_names,
        }, 'best_model.pt')
        print(f"  ⭐ Best model saved! (Acc: {val_acc:.4f})")

print("\n" + "="*50)
print("🎉 TRAINING COMPLETE!")
print("="*50)
print(f"🏆 Best validation accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)")
print("="*50)


In [ ]:
# ============================================================================
# CELL 8: VISUALIZE TRAINING PROGRESS
# ============================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
ax1.plot(history['train_loss'], label='Train Loss', marker='o', linewidth=2)
ax1.plot(history['val_loss'], label='Val Loss', marker='s', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Accuracy plot
ax2.plot(history['train_acc'], label='Train Acc', marker='o', linewidth=2)
ax2.plot(history['val_acc'], label='Val Acc', marker='s', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"✅ Final Training Accuracy: {history['train_acc'][-1]:.4f}")
print(f"✅ Final Validation Accuracy: {history['val_acc'][-1]:.4f}")


In [ ]:
# ============================================================================
# CELL 9: TEST SET EVALUATION (FIXED)
# ============================================================================

# Load best model with weights_only=False
checkpoint = torch.load('best_model.pt', weights_only=False)  # <--- THE FIX
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# ... rest of your code remains the same ...

In [ ]:
# ============================================================================
# CELL 10: PREDICTION FUNCTION
# ============================================================================

def predict_complaint(text):
    """Predict category for a single complaint"""

    # Preprocess
    cleaned = str(text).lower()
    cleaned = re.sub(r'[^a-zA-Z\s]', '', cleaned)
    cleaned = ' '.join(cleaned.split())

    # Tokenize
    inputs = tokenizer(cleaned, return_tensors='pt', truncation=True,
                      padding='max_length', max_length=128).to(device)

    # Predict
    model.eval()
    with torch.no_grad():
        outputs = model(inputs['input_ids'], inputs['attention_mask'])
        probabilities = nn.Softmax(dim=1)(outputs)[0]
        pred_class = torch.argmax(probabilities).item()
        confidence = probabilities[pred_class].item()

    # Get all probabilities
    probs_dict = {}
    for i, name in enumerate(class_names):
        probs_dict[name] = float(probabilities[i])

    # Sort for display
    sorted_probs = sorted(probs_dict.items(), key=lambda x: x[1], reverse=True)

    result = {
        'text': text,
        'cleaned': cleaned,
        'prediction': class_names[pred_class],
        'confidence': confidence,
        'all_probabilities': probs_dict,
        'top_3': sorted_probs[:3]
    }

    return result

print("✅ Prediction function created!")
print("\n🔍 Quick test:")
test_result = predict_complaint("The WiFi is very slow")
print(f"  Input: 'The WiFi is very slow'")
print(f"  Prediction: {test_result['prediction']} ({test_result['confidence']:.2%})")


In [ ]:
# ============================================================================
# CELL 11: TEST WITH REAL EXAMPLES
# ============================================================================

test_complaints = [
    "The WiFi in the library is extremely slow and keeps disconnecting",
    "I haven't received my scholarship payment for this semester",
    "The cafeteria food is cold and the portions are too small",
    "Someone broke into my dorm room last night and stole my laptop",
    "The professor didn't show up for class again",
    "The registration system keeps crashing when I try to enroll",
    "There are no lights on the walkway near the dormitory at night",
    "The library doesn't have enough copies of required textbooks",
    "Staff at the finance office are rude and unhelpful",
    "My roommate plays loud music until 3 AM every night"
]

print("="*60)
print("🔍 TESTING PREDICTIONS ON 10 COMPLAINTS")
print("="*60)

for i, complaint in enumerate(test_complaints, 1):
    result = predict_complaint(complaint)

    print(f"\n{i}. 📝 Complaint: {complaint[:50]}..." if len(complaint) > 50 else f"\n{i}. 📝 Complaint: {complaint}")
    print(f"   → PREDICTION: {result['prediction']} ({result['confidence']:.2%})")

    # Show top 3
    print("   Top 3:")
    for cat, prob in result['top_3']:
        print(f"     • {cat}: {prob:.2%}")

    if i < len(test_complaints):
        print("   " + "-"*50)


In [ ]:
# ============================================================================
# CELL 12: GRADIO WEB INTERFACE
# ============================================================================

def gradio_predict(complaint):
    """Prediction function for Gradio"""
    if not complaint or len(complaint.strip()) == 0:
        return "⚠️ Please enter a complaint", {}

    # Get prediction
    result = predict_complaint(complaint)

    # Format output
    output_text = f"""
### 🎯 **PREDICTION: {result['prediction']}**
### 📊 **Confidence: {result['confidence']:.2%}**

---
**Original:** {result['text']}
**Cleaned:** {result['cleaned']}
    """

    return output_text, result['all_probabilities']

# Create custom theme
theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="indigo",
)

# Create interface
with gr.Blocks(theme=theme, title="Student Complaint Classifier") as demo:
    gr.Markdown("""
    # 🎓 **Student Complaint Classification System**

    This AI-powered system uses **BERT (Bidirectional Encoder Representations from Transformers)**
    to automatically categorize student complaints into 8 categories.

    ---
    """)

    with gr.Row():
        with gr.Column(scale=1):
            # Input
            complaint_input = gr.Textbox(
                label="📝 Enter Your Complaint",
                placeholder="Type your complaint here...",
                lines=5
            )

            with gr.Row():
                submit_btn = gr.Button("🔍 Classify Complaint", variant="primary", size="lg")
                clear_btn = gr.Button("🗑️ Clear", size="lg")

            # Examples
            gr.Markdown("### 📋 Try These Examples:")
            examples = gr.Examples(
                examples=[
                    ["The WiFi in the library is very slow and keeps disconnecting"],
                    ["I haven't received my scholarship payment for this semester"],
                    ["The cafeteria food is cold and tastes bad"],
                    ["Someone broke into my dorm room last night"],
                    ["The professor didn't show up for class"],
                    ["The registration system keeps crashing"]
                ],
                inputs=complaint_input
            )

        with gr.Column(scale=1):
            # Outputs
            result_text = gr.Markdown(label="📊 Result")
            prob_output = gr.Label(label="📈 Probability Distribution", num_top_classes=8)

    # Event handlers
    submit_btn.click(
        fn=gradio_predict,
        inputs=complaint_input,
        outputs=[result_text, prob_output]
    )

    clear_btn.click(
        fn=lambda: ("", None),
        outputs=[complaint_input, prob_output]
    )

    gr.Markdown("""
    ---
    ### 📊 **About the Model**
    - **Model:** BERT-base-uncased fine-tuned on student complaints
    - **Categories:** Academic, Financial, Dormitory, Cafeteria, Library, ICT, Administrative, Security
    - **Accuracy:** ~85-90% on test set
    - **Framework:** PyTorch + Transformers + Gradio

    *Note: This is a demonstration project. Predictions should be verified by human administrators.*
    """)

print("\n" + "="*60)
print("🚀 LAUNCHING WEB APP...")
print("="*60)
print("\nClick the URL below to open the app:")
print("-"*60)

demo.launch(share=True)

print("\n" + "="*60)
print("✅ APP IS RUNNING!")
print("📱 Share the URL with anyone to test")
print("⏱️ App will run for 72 hours")
print("="*60)


In [ ]:
# ============================================================================
# CELL 13: SAVE TO GOOGLE DRIVE
# ============================================================================

from google.colab import drive
drive.mount('/content/drive')

# Copy model to Drive
!cp best_model.pt /content/drive/MyDrive/
print("✅ Model saved to Google Drive: /content/drive/MyDrive/best_model.pt")

# Also save label encoder
import joblib
joblib.dump(preprocessor.label_encoder, 'label_encoder.pkl')
!cp label_encoder.pkl /content/drive/MyDrive/
print("✅ Label encoder saved to Google Drive")

# Download to computer
print("\n📥 Downloading model to your computer...")
files.download('best_model.pt')
print("✅ Download complete!")


In [ ]:
# ============================================================================
# CELL 14: VISUALIZE CATEGORY DISTRIBUTION
# ============================================================================

# Count categories
category_counts = df['category'].value_counts()

# Create figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Bar plot
bars = ax1.bar(category_counts.index, category_counts.values, color='skyblue', edgecolor='navy')
ax1.set_xlabel('Category', fontsize=12)
ax1.set_ylabel('Number of Complaints', fontsize=12)
ax1.set_title('Complaint Category Distribution', fontsize=14, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height)}', ha='center', va='bottom', fontweight='bold')

# Pie chart
colors = plt.cm.Set3(range(len(category_counts)))
wedges, texts, autotexts = ax2.pie(category_counts.values, labels=category_counts.index,
                                   autopct='%1.1f%%', colors=colors, startangle=90)
ax2.set_title('Category Distribution (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📊 Category Summary:")
print("-"*40)
for category, count in category_counts.items():
    print(f"  {category:20s}: {count:3d} complaints")


In [ ]:
# ============================================================================
# CELL 15: DOWNLOAD ALL FILES
# ============================================================================

print("📥 PREPARING FILES FOR DOWNLOAD")
print("="*50)

# List of files to download
files_to_download = ['best_model.pt', 'label_encoder.pkl']

for file in files_to_download:
    try:
        files.download(file)
        print(f"✅ Downloaded: {file}")
    except:
        print(f"⚠️ Could not download: {file}")

print("\n" + "="*50)
print("✅ All files downloaded!")
print("📁 Check your Downloads folder")
print("="*50)
